In [6]:
# %pip install google-play-scraper
# %pip install pandas

In [7]:
from google_play_scraper import Sort, reviews, reviews_all, app
import json, pprint, time, re, nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [8]:
app_args = {
    "app_id": "com.btpn.dc",
    "lang": "en",
    "country": "id"
}

In [9]:
app_info = app(**app_args)

pd.DataFrame([app_info])
# print(json.dumps(app_info, indent=2))

,title,description,descriptionHTML,summary,installs,minInstalls,realInstalls,score,ratings,reviews,...,contentRatingDescription,adSupported,containsAds,released,lastUpdatedOn,updated,version,comments,appId,url
0,Jenius,The simple way to manage your life/finance wit...,The simple way to manage your life/finance wit...,The simple way to manage your life/finance wit...,"10,000,000+",10000000,13761625,3.274812,203279,129302,...,None,False,False,"Aug 10, 2016","Mar 13, 2026",1773408818,4.64.0,[],com.btpn.dc,https://play.google.com/store/apps/details?id=...


In [10]:
# pretty print 5 reviews based on 1* rating
result, continuation_token = reviews(
    **app_args,
    sort=Sort.RATING, # defaults to Sort.NEWEST
    count=5, # defaults to 100
    filter_score_with=1 # defaults to None(means all score)
)

result, _ = reviews(
    app_args['app_id'],
    continuation_token=continuation_token # defaults to None(load from the beginning)
)

pprint.pprint(result)

[{'appVersion': '4.63.1',
  'at': datetime.datetime(2026, 2, 23, 23, 5, 21),
  'content': 'It was good. but getting slower now. even scanning QRIS took '
             'longer compared to other mobile banking apps.',
  'repliedAt': datetime.datetime(2026, 2, 23, 23, 7, 19),
  'replyContent': 'Hi Benny Herlambang. We apologize for the inconvenience. '
                  'The information you provided has been noted and will be '
                  'used as feedback for Jenius to provide/improve better '
                  'services and facilities.',
  'reviewCreatedVersion': '4.63.1',
  'reviewId': '8a566d92-07ef-4275-9731-49bc9c34a2f3',
  'score': 1,
  'thumbsUpCount': 0,
  'userImage': 'https://play-lh.googleusercontent.com/a-/ALV-UjX8vrwAXJftZ8y96Nu6-Q1tOmHECQ9TuedRZLRByohZF_bVgpe0Ew',
  'userName': 'Benny Herlambang'},
 {'appVersion': '4.55.0',
  'at': datetime.datetime(2025, 12, 11, 20, 12, 22),
  'content': 'Nothing works, when i ask for help no one knows what to do, '
             'no

In [ ]:
# piles all reviews
all_reviews = []
review_ids = set()

# 1 batch = 200 reviews, 10000/200 = 50x
target_count = 10000
batch_size = 200

for sort_type in [Sort.MOST_RELEVANT, Sort.NEWEST]:
  continuation_token = None
  no_new_reviews_count = 0  # Track consecutive batches with no new reviews

  while len(all_reviews) < target_count:
    result, continuation_token = reviews(
      **app_args,
      sort=sort_type,
      count=batch_size,
      continuation_token=continuation_token
    )

    # Track count before adding
    prev_count = len(all_reviews)

    # dupe prevention
    for review in result:
      if len(all_reviews) >= target_count:
        break

      review_id = review['reviewId']
      if review_id not in review_ids:
        review_ids.add(review_id)
        all_reviews.append(review)

    # Check if any new reviews were added
    if len(all_reviews) == prev_count:
      no_new_reviews_count += 1
      if no_new_reviews_count >= 3:  # Stop after 3 consecutive batches with no new reviews
        print(f"No new unique reviews found. Stopping {sort_type}.")
        break
    else:
      no_new_reviews_count = 0  # Reset if we found new reviews

    # Stop if no more reviews avail.
    if continuation_token is None or len(result) == 0:
      break

    print(f"Scraped {len(all_reviews)} reviews in total.")
    time.sleep(4)

  if len(all_reviews) >= target_count:
    break

  print(f"Switched sorting method. Total: {len(all_reviews)}")

# Convert to DataFrame
df = pd.DataFrame(all_reviews)
print(f"Total reviews collected: {len(df)}.")


Scraped 200 reviews in total.
Scraped 400 reviews in total.
Scraped 600 reviews in total.
Scraped 800 reviews in total.
Scraped 1000 reviews in total.
Scraped 1200 reviews in total.
Scraped 1400 reviews in total.
Scraped 1600 reviews in total.
Scraped 1800 reviews in total.
Scraped 2000 reviews in total.
Scraped 2200 reviews in total.
Scraped 2400 reviews in total.
Scraped 2600 reviews in total.
Scraped 2800 reviews in total.
Scraped 3000 reviews in total.
Total reviews collected: 3000.


In [12]:
# create label sentiment from ratings
def label_sentiment(score):
  if score >= 4:
    return 'positive'
  elif score == 3:
    return 'neutral'
  else:
    return 'negative'

df['sentiment'] = df['score'].apply(label_sentiment)

print(df['sentiment'].value_counts(normalize=True))

sentiment
negative    0.657000
positive    0.242667
neutral     0.100333
Name: proportion, dtype: float64


In [13]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [14]:
def preprocess_text(text):
  text = text.lower()

  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = re.sub(r'@\w+', '', text)
  text = re.sub(r'[^a-zA-Z\s]', '', text)

  text = ' '.join(text.split())

  stop_words = set(stopwords.words('english'))
  lemmatizer = WordNetLemmatizer()

  words = text.split()
  words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

  return ' '.join(words)

df['cleaned_content'] = df['content'].apply(preprocess_text)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Create TF-IDF features
tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,      # Top 2000 most frequent words
    min_df=5,               # Word must appear in at least 5 reviews
    max_df=0.7,             # Ignore words in >70% of reviews
    ngram_range=(1, 2)      # Use unigrams and bigrams
)

# Fit and transform the data
X = tfidf_vectorizer.fit_transform(df['cleaned_content'])
y = df['sentiment']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


Training set: (2400, 2000)
Test set: (600, 2000)


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# using Logistic Regression approach
lr_model = LogisticRegression(
    C=1.0,                    # Regularization strength
    max_iter=1000,            # Sufficient iterations
    class_weight='balanced',  # Handle imbalanced data
    random_state=42
    )
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

accuracy = accuracy_score(y_test, lr_pred)
print(f"Accuracy: {accuracy:.2%}")

print(classification_report(y_test, lr_pred))

Accuracy: 75.33%
              precision    recall  f1-score   support

    negative       0.87      0.80      0.83       394
     neutral       0.24      0.42      0.31        60
    positive       0.82      0.77      0.80       146

    accuracy                           0.75       600
   macro avg       0.65      0.66      0.65       600
weighted avg       0.80      0.75      0.77       600



In [19]:
# fix for low accuracy
# Remove neutral class
df_binary = df[df['sentiment'] != 'neutral'].copy()

# Re-extract features and train
X = tfidf_vectorizer.fit_transform(df_binary['cleaned_content'])
y = df_binary['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42)

lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print(f"Binary Accuracy: {accuracy_score(y_test, lr_pred):.2%}")

print(classification_report(y_test, lr_pred))

Binary Accuracy: 88.70%
              precision    recall  f1-score   support

    negative       0.87      0.99      0.93       394
    positive       0.97      0.60      0.74       146

    accuracy                           0.89       540
   macro avg       0.92      0.80      0.84       540
weighted avg       0.90      0.89      0.88       540

